In [1]:
!pip install pyspark==3.5.3 delta-spark==3.2.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 65.5 MB/s  0:00:04
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840690 sha256=bf6ded6ecd62f8dc74a381f087a4065d21b58493962f04f3b4af7a91d55f845f
  Stored in directory: /root/.cache/pip/wheels/57/01/5d/dd344b64ae939cd8ea67ffa0abd6aed72f14b1165228e281a0
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.5
    Uninstalling pyspark-4.0.5:
      Successfully uninstalled pyspark-4.0.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [delta-spark]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("SkyPoints-Assessment")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)


Spark Version: 3.5.3


In [3]:
data = [
    (223457, "Elena", "USA"),
    (223458, "Ravi", "IND"),
    (223459, "Mateo", "PHIL")
]

columns = ["Member_ID", "Member_Name", "Country"]

df = spark.createDataFrame(data, columns)

df.show()

df.write.format("delta").mode("overwrite").save("/content/skypoints_delta")

result = spark.read.format("delta").load("/content/skypoints_delta")

result.show()

+---------+-----------+-------+
|Member_ID|Member_Name|Country|
+---------+-----------+-------+
|   223457|      Elena|    USA|
|   223458|       Ravi|    IND|
|   223459|      Mateo|   PHIL|
+---------+-----------+-------+

+---------+-----------+-------+
|Member_ID|Member_Name|Country|
+---------+-----------+-------+
|   223457|      Elena|    USA|
|   223458|       Ravi|    IND|
|   223459|      Mateo|   PHIL|
+---------+-----------+-------+



In [4]:
from pathlib import Path

base_path = Path("/content/skypoints")

folders = [
    "data/landing",
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/quarantine",
    "output",
    "logs"
]

for folder in folders:
    (base_path / folder).mkdir(parents=True, exist_ok=True)

print("SkyPoints folder structure created successfully!")

SkyPoints folder structure created successfully!


In [7]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print("Uploaded:", filename)

Saving AUS.xlsx to AUS.xlsx
Saving IND.xlsx to IND.xlsx
Saving USA.xlsx to USA.xlsx
Uploaded: AUS.xlsx
Uploaded: IND.xlsx
Uploaded: USA.xlsx


In [8]:
import shutil

landing_path = base_path / "data" / "landing"

for filename in uploaded.keys():
    shutil.move(
        f"/content/{filename}",
        str(landing_path / filename)
    )

print("Files moved to landing folder!")

Files moved to landing folder!


In [9]:
for file in landing_path.iterdir():
    print(file.name, "-", file.stat().st_size, "bytes")

IND.xlsx - 10279 bytes
USA.xlsx - 10190 bytes
AUS.xlsx - 15478 bytes
